# Estimating Indian population size
Since last census in India was in 2011, precise population currently is not known, however there are various expert guesses.

**TLDR**: Regression (assuming exponential population increase) predicted *1423.1 million*  as of 2021. This is fairly close to [this expert prediction](https://www.reuters.com/world/india/india-have-29-mln-more-people-than-china-by-mid-2023-un-estimate-shows-2023-04-19/) of *1428.6 million* as of mid-2023.

![Census plot: 1901 -> 2011](https://upload.wikimedia.org/wikipedia/commons/d/de/India_population_increase.GIF)

In [ ]:
%pip install nbformat        # plotly says mime type rendering requires nbformat>=4.2

In [18]:
import pandas as pd
from sklearn.linear_model import LinearRegression        # for some bizzarre reason, sometimes import sklearn; sklearn.linear_model doesn't work (attribute error)
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline, make_pipeline
import statsmodels
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px

In [20]:
# copied from above wikipedia image
df = pd.DataFrame([
    (1901, 238.4),
    (1911, 252.09),
    (1921, 251.32),
    (1931, 278.98),
    (1941, 318.16),
    (1951, 361.09),
    (1961, 439.23),
    (1971, 548.16),
    (1981, 683.33),
    (1991, 846.42),
    (2001, 1028.74),
    (2011, 1210.19)
], columns=['year', 'population_million'])
df = df[2:]     # drop first 2 yrs (outliers) - increase then decrease
xfuture = [2021,2031,2041]
df

,year,population_million
2,1921,251.32
3,1931,278.98
4,1941,318.16
5,1951,361.09
6,1961,439.23
7,1971,548.16
8,1981,683.33
9,1991,846.42
10,2001,1028.74
11,2011,1210.19


In [ ]:
X, y = df[['year']], df['population_million']        # training data
reg = LinearRegression().fit(X,y)
print(reg.score(X,y), reg.coef_, reg.intercept_)      # 0.9 correlation obtained is nearly perfect
yfuture = reg.predict(pd.DataFrame({'year': xfuture})
print('Predicted (future):', yfuture))
ypred = reg.predict(X)
fig = px.scatter(df, x='year', y='population_million', title='Direct Regression')
fig.add_trace(px.line(df, x='year', y=ypred).data[0])
fig.update_xaxes(tickmode='array', tickvals=df['year'])       # show exact x axis labels
fig.show()
#plt.scatter(df['year'], df['population_million'])
#plt.plot(df['year'], reg_model.predict(X))

0.9257610324940422 [10.66369697] -20368.266242424244
Predicted (future): [1183.06533333 1289.70230303 1396.33927273]


In [16]:
X, y = df[['year']], df['population_million']        # training data
reg_log = LinearRegression().fit(X, np.log(y))
print(reg.score(X,y), reg_log.coef_, reg_log.intercept_)      # 0.9 correlation obtained is nearly perfect
print('Predicted:', np.exp(reg_log.predict(pd.DataFrame({'year': [2021,2031,2041]}))))
ypred = np.exp(reg_log.predict(X))
#plt.scatter(df['year'], df['population_million'])
fig = px.scatter(df, x='year', y='population_million', title='Regression after taking log(y)')
fig.add_trace(px.line(df, x='year', y=ypred).data[0])
fig.update_xaxes(tickmode='array', tickvals=df['year'])       # show exact x axis labels
fig.show()

0.9257610324940422 [0.01836867] -29.86242627878029
Predicted: [1423.18457705 1710.1544759  2054.98877561]


In [ ]:
# residual plot: TODO


In [ ]:
# Source: https://stackoverflow.com/a/74673133/12947681
# Using statsmodels just to try to estimate confidence interval of prediction (i.e. determine upper and lower bounds of our 2021 prediction)
import statsmodels.api as sm
alpha = 0.05 # 95% confidence interval
lr = sm.OLS(y, sm.add_constant(X)).fit()
conf_interval = lr.conf_int(alpha)
conf_interval

,0,1
const,-25209.089813,-15527.442672
year,8.201689,13.125705


In [22]:
lr?

Type:            RegressionResultsWrapper
String form:     <statsmodels.regression.linear_model.RegressionResultsWrapper object at 0x7f99ea874ec0>
File:            ~/.local/lib/python3.13/site-packages/statsmodels/regression/linear_model.py
Docstring:      
Results class for for an OLS model.

Parameters
----------
model : RegressionModel
    The regression model instance.
params : ndarray
    The estimated parameters.
normalized_cov_params : ndarray
    The normalized covariance parameters.
scale : float
    The estimated scale of the residuals.
cov_type : str
    The covariance estimator used in the results.
cov_kwds : dict
    Additional keywords used in the covariance specification.
use_t : bool
    Flag indicating to use the Student's t in inference.
**kwargs
    Additional keyword arguments used to initialize the results.

See Also
--------
RegressionResults
    Results store for WLS and GLW models.

Notes
-----
Most of the methods and attributes are inherited from RegressionResu

In [23]:
lr.model?

Type:        OLS
String form: <statsmodels.regression.linear_model.OLS object at 0x7f99ea874ad0>
File:        ~/.local/lib/python3.13/site-packages/statsmodels/regression/linear_model.py
Docstring:  
Ordinary Least Squares

Parameters
----------
endog : array_like
    A 1-d endogenous response variable. The dependent variable.
exog : array_like
    A nobs x k array where `nobs` is the number of observations and `k`
    is the number of regressors. An intercept is not included by default
    and should be added by the user. See
    :func:`statsmodels.tools.add_constant`.
missing : str
    Available options are 'none', 'drop', and 'raise'. If 'none', no nan
    checking is done. If 'drop', any observations with nans are dropped.
    If 'raise', an error is raised. Default is 'none'.
hasconst : None or bool
    Indicates whether the RHS includes a user-supplied constant. If True,
    a constant is not checked for and k_constant is set to 1 and all
    result statistics are calculated as i